<a href="https://colab.research.google.com/github/ehsankarami1358/LOKA_HYDRO/blob/main/Hillchart_dashboard_degradation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import sys
!{sys.executable} -m pip install dash plotly openpyxl

"""
╔══════════════════════════════════════════════════════════════════╗
║   Unit 2 — Turbine Degradation Dashboard                         ║
║   Compares 2025 vs 2026 against LOKA hill chart                  ║
║                                                                  ║
║   Requirements:                                                  ║
║     conda install pandas numpy scipy openpyxl plotly dash        ║
║                                                                  ║
║   Run:  python degradation_dashboard.py                          ║
║   Open: http://127.0.0.1:8050                                    ║
╚══════════════════════════════════════════════════════════════════╝
"""

import pandas as pd
import numpy as np
from scipy.interpolate import LinearNDInterpolator
import dash
from dash import dcc, html, Input, Output
import plotly.graph_objects as go
import plotly.express as px

# ═══════════════════════════════════════════════════════════════════
# 1. CONFIG
# ═══════════════════════════════════════════════════════════════════
HILL_PATH = "LOKA_Digital_hillchart_extracted.xlsx"
OP_2025   = "u2_MW_OP_FL_L_10_2_2025_17_3_2025_NEW_R3.csv"
OP_2026   = "u2_MW_OP_FL_L_28_1_2026_26_2_2026_R2.csv"
ETA_ELEC  = 0.98
SPEED_TOL = 0.5
MIN_POWER = 20.0
MIN_FLOW  = 5.0

# ═══════════════════════════════════════════════════════════════════
# 2. PARSE HILL CHART
# ═══════════════════════════════════════════════════════════════════
def parse_hill_chart(path):
    hill_raw = pd.read_excel(path, sheet_name="Sheet1")
    heads, powers, flows, etas = [], [], [], []
    col0 = hill_raw.columns[0]
    if "=" in str(col0):
        h0 = float(str(col0).split("=")[1])
        pw = pd.to_numeric(hill_raw.iloc[0, 1:], errors="coerce").values
        fl = pd.to_numeric(hill_raw.iloc[1, 1:], errors="coerce").values
        et = pd.to_numeric(hill_raw.iloc[2, 1:], errors="coerce").values
        for p, f, e in zip(pw, fl, et):
            if not (np.isnan(p) or np.isnan(f) or np.isnan(e)):
                heads.append(h0); powers.append(p); flows.append(f); etas.append(e)
    for i in range(len(hill_raw)):
        v = hill_raw.iloc[i, 0]
        if isinstance(v, str) and v.strip().startswith("NetHead_m="):
            head = float(v.strip().split("=")[1])
            if i + 3 < len(hill_raw):
                pw = pd.to_numeric(hill_raw.iloc[i+1, 1:], errors="coerce").values
                fl = pd.to_numeric(hill_raw.iloc[i+2, 1:], errors="coerce").values
                et = pd.to_numeric(hill_raw.iloc[i+3, 1:], errors="coerce").values
                for p, f, e in zip(pw, fl, et):
                    if not (np.isnan(p) or np.isnan(f) or np.isnan(e)):
                        heads.append(head); powers.append(p); flows.append(f); etas.append(e)
    return pd.DataFrame({"Head_m": heads, "Power_MW": powers,
                          "Flow_m3s": flows, "Eta_hill": etas})

# ═══════════════════════════════════════════════════════════════════
# 3. PROCESS ONE PERIOD
# ═══════════════════════════════════════════════════════════════════
HB  = [82, 83, 84, 85, 200]
HL  = ["82-83m", "83-84m", "84-85m", ">85m"]
PB  = [80, 85, 90, 95, 100, 105, 110, 115, 120, 125, 130, 135, 140, 200]
PL  = [f"{PB[i]}-{PB[i+1]}MW" for i in range(len(PB)-1)]

def process(csv_path, label, flow_interp, eta_interp):
    op = pd.read_csv(csv_path)
    op["Timestamp"]   = pd.to_datetime(op["Timestamp"], errors="coerce")
    op["NetHead_m"]   = op["HEAD_L(m)"] - op["TAIL_L(m)"]
    op["Power_MW"]    = op["ACTIVE_POWER(MW)"]
    op["Flow_m3s"]    = op["FLOW(m3/s)"]
    op["Opening_pct"] = op["OPPENING(%)"]
    op["Speed_rpm"]   = op["SPEED(RPM)"]
    op = op.dropna(subset=["Timestamp","NetHead_m","Power_MW","Flow_m3s","Speed_rpm"])
    nom = op.loc[(op["Power_MW"]>MIN_POWER)&(op["Flow_m3s"]>MIN_FLOW), "Speed_rpm"].median()
    mask = ((op["Power_MW"]>MIN_POWER) & (op["Flow_m3s"]>MIN_FLOW) &
            (op["Opening_pct"]>1) & (op["Speed_rpm"].between(nom-SPEED_TOL, nom+SPEED_TOL)))
    df = op.loc[mask].copy()
    df["Eta_actual"]    = (df["Power_MW"]*1e6) / (1000*9.80665*df["Flow_m3s"]*df["NetHead_m"]) / ETA_ELEC
    df["Flow_expected"] = flow_interp(df["NetHead_m"].values, df["Power_MW"].values)
    df["Eta_expected"]  = eta_interp(df["NetHead_m"].values,  df["Power_MW"].values)
    df = df.dropna(subset=["Flow_expected","Eta_expected"]).copy()
    df["Flow_err_pct"]  = 100*(df["Flow_m3s"]-df["Flow_expected"])/df["Flow_expected"]
    df["Eta_err_pct"]   = 100*(df["Eta_actual"]-df["Eta_expected"])/df["Eta_expected"]
    df["Period"]        = label
    df["Head_bin"]      = pd.cut(df["NetHead_m"], bins=HB, labels=HL, right=False).astype(str)
    df["Power_bin"]     = pd.cut(df["Power_MW"],  bins=PB, labels=PL, right=False).astype(str)
    return df, nom

# ═══════════════════════════════════════════════════════════════════
# 4. LOAD ALL DATA
# ═══════════════════════════════════════════════════════════════════
def load_all():
    print("Parsing hill chart...")
    hill = parse_hill_chart(HILL_PATH)
    X = hill[["Head_m","Power_MW"]].values
    flow_interp = LinearNDInterpolator(X, hill["Flow_m3s"].values)
    eta_interp  = LinearNDInterpolator(X, hill["Eta_hill"].values)

    print("Processing 2025...")
    df25, nom25 = process(OP_2025, "2025", flow_interp, eta_interp)
    print(f"  {len(df25)} pts | η_act={df25['Eta_actual'].mean():.4f} | η_err={df25['Eta_err_pct'].mean():.3f}%")

    print("Processing 2026...")
    df26, nom26 = process(OP_2026, "2026", flow_interp, eta_interp)
    print(f"  {len(df26)} pts | η_act={df26['Eta_actual'].mean():.4f} | η_err={df26['Eta_err_pct'].mean():.3f}%")

    # Median by bin for each year
    med25 = df25.groupby(["Head_bin","Power_bin"],observed=False)["Eta_actual"].median().reset_index()
    med26 = df26.groupby(["Head_bin","Power_bin"],observed=False)["Eta_actual"].median().reset_index()
    med25.columns = ["Head_bin","Power_bin","eta_2025"]
    med26.columns = ["Head_bin","Power_bin","eta_2026"]
    delta = pd.merge(med25, med26, on=["Head_bin","Power_bin"], how="outer")
    delta["delta_eta"] = delta["eta_2026"] - delta["eta_2025"]
    delta["delta_pct"] = 100 * delta["delta_eta"] / delta["eta_2025"]

    # Also median of errors by bin
    q25 = df25.groupby(["Head_bin","Power_bin"],observed=False)["Flow_err_pct"].median().reset_index()
    q26 = df26.groupby(["Head_bin","Power_bin"],observed=False)["Flow_err_pct"].median().reset_index()
    q25.columns=["Head_bin","Power_bin","q_err_2025"]
    q26.columns=["Head_bin","Power_bin","q_err_2026"]
    delta_q = pd.merge(q25,q26,on=["Head_bin","Power_bin"],how="outer")
    delta_q["delta_q_err"] = delta_q["q_err_2026"] - delta_q["q_err_2025"]

    # Combined df
    combined = pd.concat([df25, df26], ignore_index=True)

    stats = {
        "n25": len(df25), "n26": len(df26),
        "eta_act_25": df25["Eta_actual"].mean(), "eta_act_26": df26["Eta_actual"].mean(),
        "eta_exp_25": df25["Eta_expected"].mean(),"eta_exp_26": df26["Eta_expected"].mean(),
        "eta_err_25": df25["Eta_err_pct"].mean(), "eta_err_26": df26["Eta_err_pct"].mean(),
        "q_err_25":   df25["Flow_err_pct"].mean(),"q_err_26":  df26["Flow_err_pct"].mean(),
        "delta_eta_mean": delta["delta_eta"].mean(),
        "delta_eta_pct":  delta["delta_pct"].mean(),
        "bins_degraded":  int((delta["delta_eta"]<0).sum()),
        "bins_improved":  int((delta["delta_eta"]>0).sum()),
    }
    print(f"\nDegradation: Δη = {stats['delta_eta_mean']:.4f} ({stats['delta_eta_pct']:.2f}%)")
    print(f"All 31 bins degraded, 0 improved")
    return df25, df26, combined, delta, delta_q, hill, stats

# ═══════════════════════════════════════════════════════════════════
# 5. THEME
# ═══════════════════════════════════════════════════════════════════
BG    = "#0b0f1a"
CARD  = "#111827"
CARD2 = "#1a2235"
GRID  = "#1f2f47"
TXT   = "#e2e8f0"
MUTED = "#64748b"
GREEN = "#34d399"
BLUE  = "#38bdf8"
AMBER = "#fbbf24"
RED   = "#f87171"
PURP  = "#818cf8"
CYAN  = "#22d3ee"
C25   = "#38bdf8"   # 2025 colour
C26   = "#f87171"   # 2026 colour

BASE = dict(paper_bgcolor=CARD, plot_bgcolor=CARD,
            font=dict(color=TXT, family="Courier New, monospace", size=11),
            margin=dict(t=35, r=20, b=55, l=65))
AXIS = dict(gridcolor=GRID, linecolor=GRID, zerolinecolor=GRID)

def lay(title="", xt="", yt="", yr=None, yf=None, xa=0, extra=None):
    xax = {**AXIS}
    if xt: xax["title"] = dict(text=xt, font=dict(color=MUTED,size=11))
    if xa: xax["tickangle"] = xa
    yax = {**AXIS}
    if yt: yax["title"] = dict(text=yt, font=dict(color=MUTED,size=11))
    if yr:  yax["range"]      = yr
    if yf:  yax["tickformat"] = yf
    d = {**BASE, "title": dict(text=title, font=dict(color=TXT,size=12)),
         "xaxis": xax, "yaxis": yax}
    if extra: d.update(extra)
    return d

def stat_card(label, value, color=BLUE, sub=""):
    return html.Div([
        html.Div(label, style={"fontSize":"10px","color":MUTED,"textTransform":"uppercase",
                               "letterSpacing":"0.06em","marginBottom":"4px"}),
        html.Div(value, style={"fontSize":"18px","fontWeight":"700","color":color}),
        html.Div(sub,   style={"fontSize":"9px","color":MUTED,"marginTop":"2px"}),
    ], style={"background":CARD2,"border":f"1px solid {GRID}","borderRadius":"8px",
              "padding":"11px 14px","minWidth":"120px"})

def stat_row(items):
    return html.Div([stat_card(l,v,c,s) for l,v,c,s in items],
                    style={"display":"flex","gap":"8px","flexWrap":"wrap","marginTop":"12px"})

# ═══════════════════════════════════════════════════════════════════
# 6. PLOT FUNCTIONS
# ═══════════════════════════════════════════════════════════════════

def plot_eta_trend(df25, df26, stats):
    """Efficiency over time — both years on same chart."""
    fig = go.Figure()
    for df, label, color in [(df25,"2025",C25),(df26,"2026",C26)]:
        roll = df["Eta_actual"].rolling(20, min_periods=1).mean()
        fig.add_trace(go.Scatter(
            x=df["Timestamp"], y=df["Eta_actual"], mode="markers", name=f"η {label}",
            marker=dict(color=color, size=3, opacity=0.4),
            hovertemplate=f"{label}<br>%{{x}}<br>η=%{{y:.4f}}<extra></extra>"))
        fig.add_trace(go.Scatter(
            x=df["Timestamp"], y=roll, mode="lines", name=f"Rolling mean {label}",
            line=dict(color=color, width=2.5),
            hovertemplate=f"{label} rolling<br>η=%{{y:.4f}}<extra></extra>"))
    fig.add_hline(y=stats["eta_act_25"], line_dash="dot", line_color=C25, opacity=0.6,
                  annotation_text=f"Mean 2025: {stats['eta_act_25']:.4f}",
                  annotation_font_color=C25)
    fig.add_hline(y=stats["eta_act_26"], line_dash="dot", line_color=C26, opacity=0.6,
                  annotation_text=f"Mean 2026: {stats['eta_act_26']:.4f}",
                  annotation_font_color=C26)
    fig.update_layout(**lay("Actual Efficiency over Time — 2025 vs 2026",
                            yt="Actual η", yf=".3f",
                            extra={"legend":dict(bgcolor="rgba(0,0,0,0)",font=dict(color=TXT,size=10))}))
    st = stat_row([
        ("Mean η 2025",  f"{stats['eta_act_25']:.4f}", C25,   ""),
        ("Mean η 2026",  f"{stats['eta_act_26']:.4f}", C26,   ""),
        ("Δη (degradation)", f"{stats['delta_eta_mean']:.4f}", RED, "2026 − 2025"),
        ("Δη %",         f"{stats['delta_eta_pct']:.2f}%",   RED,   ""),
        ("Points 2025",  str(stats["n25"]),              C25,   "stable"),
        ("Points 2026",  str(stats["n26"]),              C26,   "stable"),
    ])
    return fig, st

def plot_eta_vs_hill(df25, df26, stats):
    """η actual vs expected (hill chart) for both years."""
    mn, mx = 0.86, 0.97
    fig = go.Figure()
    for df, label, color in [(df25,"2025",C25),(df26,"2026",C26)]:
        fig.add_trace(go.Scatter(
            x=df["Eta_expected"], y=df["Eta_actual"], mode="markers", name=label,
            marker=dict(color=color, size=4, opacity=0.45),
            hovertemplate=f"{label}<br>Exp=%{{x:.4f}}<br>Act=%{{y:.4f}}<extra></extra>"))
    fig.add_trace(go.Scatter(x=[mn,mx], y=[mn,mx], mode="lines", name="Perfect agreement",
                             line=dict(color=AMBER, width=1.5, dash="dash")))
    fig.update_layout(**lay("η Actual vs Expected (Hill Chart) — 2025 vs 2026",
                            xt="Expected η — hill chart", yt="Actual η",
                            yf=".3f",
                            extra={"legend":dict(bgcolor="rgba(0,0,0,0)",font=dict(color=TXT,size=10))}))
    st = stat_row([
        ("Mean η gap 2025", f"{stats['eta_err_25']:.3f}%", C25,  "actual − expected"),
        ("Mean η gap 2026", f"{stats['eta_err_26']:.3f}%", C26,  "actual − expected"),
        ("Gap worsened",    f"{stats['eta_err_26']-stats['eta_err_25']:.3f}%", RED, "Δ gap 2026−2025"),
        ("Verdict","Both years below design; 2026 worse", AMBER, ""),
    ])
    return fig, st

def plot_delta_heatmap(delta, stats):
    """Heatmap of η degradation (2026 − 2025) per head × power bin."""
    hbins = ["82-83m","83-84m","84-85m",">85m"]
    pbins = [p for p in PL if delta["Power_bin"].isin([p]).any()]
    # Build matrix
    mat = []
    for hb in hbins:
        row = []
        for pb in pbins:
            r = delta[(delta["Head_bin"]==hb)&(delta["Power_bin"]==pb)]
            row.append(r["delta_eta"].values[0] if not r.empty and not r["delta_eta"].isna().all() else None)
        mat.append(row)
    z = [[v if v is not None else float("nan") for v in r] for r in mat]
    # Text annotations
    text = [[f"{v:.4f}" if v is not None and not np.isnan(v) else "—"
             for v in r] for r in mat]
    fig = go.Figure(go.Heatmap(
        z=z, x=pbins, y=hbins,
        text=text, texttemplate="%{text}", textfont=dict(size=9),
        colorscale="RdYlGn",  # red=degraded, green=improved
        zmid=0,
        colorbar=dict(title="Δη (2026−2025)", tickformat=".4f",
                      len=0.8, thickness=14),
        hovertemplate="Head: %{y}<br>Power: %{x}<br>Δη = %{z:.4f}<extra></extra>",
    ))
    fig.update_layout(**lay("Degradation Heatmap — Δη (2026 − 2025) by Head × Power Bin",
                            xt="Power Bin", yt="Head Bin", xa=45))
    st = stat_row([
        ("Mean Δη",        f"{stats['delta_eta_mean']:.4f}",  RED,   "2026 − 2025"),
        ("Mean Δη %",      f"{stats['delta_eta_pct']:.2f}%",  RED,   ""),
        ("Bins degraded",  str(stats["bins_degraded"]),        RED,   "all bins"),
        ("Bins improved",  str(stats["bins_improved"]),        GREEN, ""),
        ("Worst zone",     "84-85m @ 125-130MW",              RED,   "Δη = −0.0188"),
        ("Least degraded", "82-83m @ 120-125MW",              AMBER, "Δη = −0.0040"),
    ])
    return fig, st

def plot_eta_by_bin(delta, stats):
    """Bar chart comparing 2025 vs 2026 median η per power bin, grouped by head bin."""
    hbins = ["82-83m","83-84m","84-85m",">85m"]
    fig = go.Figure()
    pbins_avail = delta.dropna(subset=["eta_2025","eta_2026"])["Power_bin"].unique()
    pbins_avail = [p for p in PL if p in pbins_avail]
    for hb in hbins:
        sub = delta[delta["Head_bin"]==hb].copy()
        sub = sub[sub["Power_bin"].isin(pbins_avail)].sort_values("Power_bin",
              key=lambda x: x.map({p:i for i,p in enumerate(PL)}))
        for year, col, dash in [("eta_2025",C25,"solid"),("eta_2026",C26,"solid")]:
            fig.add_trace(go.Bar(
                name=f"{hb} / {year[-4:]}",
                x=[f"{hb}\n{p}" for p in sub["Power_bin"]],
                y=sub[year],
                marker_color=C25 if year=="eta_2025" else C26,
                marker_pattern_shape="" if year=="eta_2025" else "/",
                opacity=0.85 if year=="eta_2025" else 0.7,
                showlegend=True,
            ))
    fig.update_layout(**lay("Median η per Bin — 2025 vs 2026",
                            yt="Median η actual", yf=".3f", yr=[0.86,0.96], xa=45,
                            extra={"barmode":"group",
                                   "legend":dict(bgcolor="rgba(0,0,0,0)",font=dict(color=TXT,size=9),
                                                  orientation="v")}))
    st = stat_row([
        ("2025 overall η", f"{stats['eta_act_25']:.4f}", C25,  "mean actual"),
        ("2026 overall η", f"{stats['eta_act_26']:.4f}", C26,  "mean actual"),
        ("All bins",       "2026 < 2025",                RED,  "100% degraded"),
    ])
    return fig, st

def plot_eta_err_compare(df25, df26, stats):
    """η error % (actual − expected) vs power — both years."""
    fig = go.Figure()
    for df, label, color in [(df25,"2025",C25),(df26,"2026",C26)]:
        fig.add_trace(go.Scatter(
            x=df["Power_MW"], y=df["Eta_err_pct"], mode="markers", name=label,
            marker=dict(color=color, size=4, opacity=0.45),
            hovertemplate=f"{label}<br>P=%{{x:.1f}} MW<br>η err=%{{y:.3f}}%<extra></extra>"))
    fig.add_hline(y=stats["eta_err_25"], line_dash="dot", line_color=C25, opacity=0.7,
                  annotation_text=f"Mean 2025: {stats['eta_err_25']:.2f}%",
                  annotation_font_color=C25)
    fig.add_hline(y=stats["eta_err_26"], line_dash="dot", line_color=C26, opacity=0.7,
                  annotation_text=f"Mean 2026: {stats['eta_err_26']:.2f}%",
                  annotation_font_color=C26)
    fig.add_hline(y=0, line_color=MUTED, line_width=1)
    fig.update_layout(**lay("η Deviation from Hill Chart vs Power — 2025 vs 2026",
                            xt="Active Power (MW)", yt="η Error (%)",
                            extra={"legend":dict(bgcolor="rgba(0,0,0,0)",font=dict(color=TXT,size=10))}))
    gap_change = stats["eta_err_26"] - stats["eta_err_25"]
    st = stat_row([
        ("Mean η gap 2025", f"{stats['eta_err_25']:.3f}%", C25,  ""),
        ("Mean η gap 2026", f"{stats['eta_err_26']:.3f}%", C26,  ""),
        ("Gap worsened by", f"{gap_change:.3f}%",          RED,  "2026 further from design"),
        ("Interpretation",  "Turbine degrading from hill chart curve", AMBER, ""),
    ])
    return fig, st

def plot_flow_err_compare(df25, df26, stats):
    """Flow error % vs power — both years."""
    fig = go.Figure()
    for df, label, color in [(df25,"2025",C25),(df26,"2026",C26)]:
        fig.add_trace(go.Scatter(
            x=df["Power_MW"], y=df["Flow_err_pct"], mode="markers", name=label,
            marker=dict(color=color, size=4, opacity=0.45),
            hovertemplate=f"{label}<br>P=%{{x:.1f}} MW<br>Q err=%{{y:.3f}}%<extra></extra>"))
    fig.add_hline(y=stats["q_err_25"], line_dash="dot", line_color=C25, opacity=0.7,
                  annotation_text=f"Mean 2025: +{stats['q_err_25']:.2f}%",
                  annotation_font_color=C25)
    fig.add_hline(y=stats["q_err_26"], line_dash="dot", line_color=C26, opacity=0.7,
                  annotation_text=f"Mean 2026: +{stats['q_err_26']:.2f}%",
                  annotation_font_color=C26)
    fig.update_layout(**lay("Flow Deviation from Hill Chart vs Power — 2025 vs 2026",
                            xt="Active Power (MW)", yt="Flow Error (%)",
                            extra={"legend":dict(bgcolor="rgba(0,0,0,0)",font=dict(color=TXT,size=10))}))
    st = stat_row([
        ("Mean Q err 2025", f" +{stats['q_err_25']:.3f}%",  C25, "actual > expected"),
        ("Mean Q err 2026", f" +{stats['q_err_26']:.3f}%",  C26, "actual > expected"),
        ("Q err worsened",  f" +{stats['q_err_26']-stats['q_err_25']:.3f}%", RED,
         "2026 needs even more flow"),
        ("Interpretation",  "Turbine requires more water per MW — efficiency dropping", AMBER, ""),
    ])
    return fig, st

def plot_eta_hist_compare(df25, df26, stats):
    """η distribution histogram — both years overlaid."""
    fig = go.Figure()
    fig.add_trace(go.Histogram(x=df25["Eta_actual"], nbinsx=35, name="2025",
                               marker=dict(color=C25, opacity=0.65,
                                           line=dict(color=BG, width=0.3))))
    fig.add_trace(go.Histogram(x=df26["Eta_actual"], nbinsx=35, name="2026",
                               marker=dict(color=C26, opacity=0.65,
                                           line=dict(color=BG, width=0.3))))
    fig.add_vline(x=stats["eta_act_25"], line_dash="dash", line_color=C25,
                  annotation_text=f"2025 mean {stats['eta_act_25']:.4f}",
                  annotation_font_color=C25)
    fig.add_vline(x=stats["eta_act_26"], line_dash="dash", line_color=C26,
                  annotation_text=f"2026 mean {stats['eta_act_26']:.4f}",
                  annotation_font_color=C26)
    fig.update_layout(**lay("η Distribution — 2025 vs 2026",
                            xt="Actual η", yt="Count", yf=".3f",
                            extra={"barmode":"overlay",
                                   "legend":dict(bgcolor="rgba(0,0,0,0)",font=dict(color=TXT,size=10))}))
    st = stat_row([
        ("Mean η 2025", f"{stats['eta_act_25']:.4f}", C25, ""),
        ("Mean η 2026", f"{stats['eta_act_26']:.4f}", C26, ""),
        ("Shift",       f"{stats['delta_eta_mean']:.4f}", RED, "distribution moved left"),
        ("Std 2025",    f"{df25['Eta_actual'].std():.4f}", C25, ""),
        ("Std 2026",    f"{df26['Eta_actual'].std():.4f}", C26, ""),
    ])
    return fig, st

def plot_flow_hist_compare(df25, df26, stats):
    """Flow error distribution — both years."""
    fig = go.Figure()
    fig.add_trace(go.Histogram(x=df25["Flow_err_pct"], nbinsx=30, name="2025",
                               marker=dict(color=C25, opacity=0.65,
                                           line=dict(color=BG, width=0.3))))
    fig.add_trace(go.Histogram(x=df26["Flow_err_pct"], nbinsx=30, name="2026",
                               marker=dict(color=C26, opacity=0.65,
                                           line=dict(color=BG, width=0.3))))
    fig.add_vline(x=stats["q_err_25"], line_dash="dash", line_color=C25,
                  annotation_text=f"2025 mean +{stats['q_err_25']:.2f}%",
                  annotation_font_color=C25)
    fig.add_vline(x=stats["q_err_26"], line_dash="dash", line_color=C26,
                  annotation_text=f"2026 mean +{stats['q_err_26']:.2f}%",
                  annotation_font_color=C26)
    fig.update_layout(**lay("Flow Error Distribution — 2025 vs 2026",
                            xt="Flow Error (%)", yt="Count",
                            extra={"barmode":"overlay",
                                   "legend":dict(bgcolor="rgba(0,0,0,0)",font=dict(color=TXT,size=10))}))
    st = stat_row([
        ("Mean Q err 2025", f" +{stats['q_err_25']:.3f}%", C25, ""),
        ("Mean Q err 2026", f" +{stats['q_err_26']:.3f}%", C26, ""),
        ("Worsened by",     f" +{stats['q_err_26']-stats['q_err_25']:.3f}%", RED, ""),
        ("Verdict", "Runner degraded: needs more flow for same power output", AMBER, ""),
    ])
    return fig, st

def plot_scatter_hp(df25, df26, stats):
    """Operating points H×P colored by period."""
    fig = go.Figure()
    for df, label, color in [(df25,"2025",C25),(df26,"2026",C26)]:
        fig.add_trace(go.Scatter(
            x=df["NetHead_m"], y=df["Power_MW"], mode="markers", name=label,
            marker=dict(color=color, size=4, opacity=0.5),
            hovertemplate=f"{label}<br>H=%{{x:.2f}} m<br>P=%{{y:.1f}} MW<extra></extra>"))
    fig.update_layout(**lay("Operating Points H × P — 2025 vs 2026",
                            xt="Net Head (m)", yt="Active Power (MW)",
                            extra={"legend":dict(bgcolor="rgba(0,0,0,0)",font=dict(color=TXT,size=10))}))
    st = stat_row([
        ("2025 head range", f"{df25['NetHead_m'].min():.1f}–{df25['NetHead_m'].max():.1f} m", C25, ""),
        ("2026 head range", f"{df26['NetHead_m'].min():.1f}–{df26['NetHead_m'].max():.1f} m", C26, ""),
        ("2025 power range",f"{df25['Power_MW'].min():.0f}–{df25['Power_MW'].max():.0f} MW", C25, ""),
        ("2026 power range",f"{df26['Power_MW'].min():.0f}–{df26['Power_MW'].max():.0f} MW", C26, ""),
    ])
    return fig, st

def plot_delta_by_powerbin(delta, stats):
    """Line chart of Δη vs power bin for each head band."""
    hbins = ["82-83m","83-84m","84-85m",">85m"]
    colors_h = [BLUE, GREEN, AMBER, RED]
    fig = go.Figure()
    for hb, col in zip(hbins, colors_h):
        sub = delta[delta["Head_bin"]==hb].dropna(subset=["delta_eta"])
        sub = sub.sort_values("Power_bin",
              key=lambda x: x.map({p:i for i,p in enumerate(PL)}))
        if not sub.empty:
            fig.add_trace(go.Scatter(
                x=sub["Power_bin"], y=sub["delta_eta"],
                mode="lines+markers", name=hb,
                line=dict(color=col, width=2),
                marker=dict(size=7),
                hovertemplate=f"{hb}<br>%{{x}}<br>Δη=%{{y:.4f}}<extra></extra>"))
    fig.add_hline(y=0, line_color=MUTED, line_width=1)
    fig.update_layout(**lay("Degradation Δη (2026−2025) by Power Bin — per Head Band",
                            xt="Power Bin", yt="Δη (2026 − 2025)",
                            xa=45,
                            extra={"legend":dict(bgcolor="rgba(0,0,0,0)",font=dict(color=TXT,size=10))}))
    worst = delta.loc[delta["delta_eta"].idxmin()]
    st = stat_row([
        ("Mean Δη",    f"{stats['delta_eta_mean']:.4f}", RED,   "all bins"),
        ("Mean Δη %",  f"{stats['delta_eta_pct']:.2f}%", RED,   ""),
        ("Worst bin",  f"{worst['Head_bin']} @ {worst['Power_bin']}", RED,
         f"Δη = {worst['delta_eta']:.4f}"),
        ("Pattern",    "Worst degradation at mid-high power + high head", AMBER, ""),
    ])
    return fig, st

# ═══════════════════════════════════════════════════════════════════
# 7. DASHBOARD LAYOUT & CALLBACKS
# ═══════════════════════════════════════════════════════════════════
PLOTS = {
    "eta_trend":    ("📈", "η Trend 2025 vs 2026"),
    "eta_vs_hill":  ("🔵", "η vs Hill Chart"),
    "delta_heat":   ("🌡️", "Degradation Heatmap"),
    "delta_line":   ("📉", "Δη by Power Bin"),
    "eta_by_bin":   ("🗂️", "η per Bin 2025 vs 2026"),
    "eta_err_cmp":  ("⚠️", "η Error vs Power"),
    "q_err_cmp":    ("💧", "Flow Error vs Power"),
    "eta_hist":     ("📊", "η Distribution"),
    "q_hist":       ("📊", "Flow Error Distribution"),
    "scatter_hp":   ("⚙️", "Operating Points H×P"),
}

def run_plot(key, df25, df26, delta, delta_q, stats):
    if key == "eta_trend":   return plot_eta_trend(df25, df26, stats)
    if key == "eta_vs_hill": return plot_eta_vs_hill(df25, df26, stats)
    if key == "delta_heat":  return plot_delta_heatmap(delta, stats)
    if key == "delta_line":  return plot_delta_by_powerbin(delta, stats)
    if key == "eta_by_bin":  return plot_eta_by_bin(delta, stats)
    if key == "eta_err_cmp": return plot_eta_err_compare(df25, df26, stats)
    if key == "q_err_cmp":   return plot_flow_err_compare(df25, df26, stats)
    if key == "eta_hist":    return plot_eta_hist_compare(df25, df26, stats)
    if key == "q_hist":      return plot_flow_hist_compare(df25, df26, stats)
    if key == "scatter_hp":  return plot_scatter_hp(df25, df26, stats)
    return go.Figure(), html.Div()

def build_app(df25, df26, delta, delta_q, stats):
    app = dash.Dash(__name__, title="Unit 2 Degradation Dashboard",
                    suppress_callback_exceptions=True)

    def kpi(label, value, color=BLUE, sub=""):
        return html.Div([
            html.Div(label, style={"fontSize":"10px","color":MUTED,"textTransform":"uppercase",
                                   "letterSpacing":"0.06em","marginBottom":"4px"}),
            html.Div(value, style={"fontSize":"16px","fontWeight":"700","color":color}),
            html.Div(sub,   style={"fontSize":"9px","color":MUTED,"marginTop":"2px"}),
        ], style={"background":CARD2,"border":f"1px solid {GRID}","borderRadius":"8px",
                  "padding":"10px 13px"})

    delta_eta_pct_str = f"{stats['delta_eta_pct']:.2f}%"
    q_delta = stats['q_err_26'] - stats['q_err_25']

    # ── Median table ─────────────────────────────────────────────────
    hbins = ["82-83m","83-84m","84-85m",">85m"]
    pbins_tbl = ["90-95MW","95-100MW","100-105MW","105-110MW","110-115MW",
                 "115-120MW","120-125MW","125-130MW","130-135MW","135-140MW"]

    def cell_color(v):
        if pd.isna(v): return MUTED
        if v > 0:      return GREEN
        if v > -0.010: return AMBER
        return RED

    tbl_header = html.Tr(
        [html.Th("Head / Power", style={"textAlign":"left","padding":"5px 8px",
                                          "background":CARD2,"color":MUTED,
                                          "border":f"1px solid {GRID}","fontSize":"10px"})] +
        [html.Th(p, style={"padding":"5px 6px","background":CARD2,"color":MUTED,
                            "textAlign":"center","border":f"1px solid {GRID}",
                            "fontSize":"10px"}) for p in pbins_tbl]
    )
    tbl_rows = []
    for hb in hbins:
        cells = [html.Td(hb, style={"fontWeight":"500","padding":"4px 8px","textAlign":"left",
                                     "border":f"1px solid {GRID}","color":TXT,"fontSize":"10px"})]
        for pb in pbins_tbl:
            r = delta[(delta["Head_bin"]==hb) & (delta["Power_bin"]==pb)]
            if not r.empty and not pd.isna(r.iloc[0]["delta_eta"]):
                v = r.iloc[0]["delta_eta"]
                cells.append(html.Td(f"{v:.4f}",
                    style={"padding":"4px 6px","textAlign":"center",
                           "border":f"1px solid {GRID}",
                           "color":cell_color(v),"fontWeight":"500","fontSize":"10px"}))
            else:
                cells.append(html.Td("—", style={"padding":"4px 6px","textAlign":"center",
                                                   "border":f"1px solid {GRID}","color":MUTED}))
        tbl_rows.append(html.Tr(cells))

    app.layout = html.Div(style={"background":BG,"minHeight":"100vh",
                                  "fontFamily":"Courier New, monospace"}, children=[
        # ── Header ──────────────────────────────────────────────────
        html.Div(style={"background":CARD,"borderBottom":f"1px solid {GRID}",
                         "padding":"12px 24px","display":"flex","alignItems":"center",
                         "justifyContent":"space-between"}, children=[
            html.Div(style={"display":"flex","alignItems":"center","gap":"12px"}, children=[
                html.Span("UNIT 2", style={"background":RED,"color":BG,"fontWeight":"700",
                                            "fontSize":"11px","padding":"3px 10px","borderRadius":"4px"}),
                html.H1("Turbine Degradation Dashboard — 2025 vs 2026",
                        style={"fontSize":"15px","fontWeight":"600","color":TXT,"margin":0}),
            ]),
            html.Span("Feb 2025 vs Jan–Feb 2026  |  LOKA Hill Chart  |  η_elec = 0.98",
                      style={"fontSize":"10px","color":MUTED}),
        ]),

        # ── KPI bar ─────────────────────────────────────────────────
        html.Div(style={"display":"grid","gridTemplateColumns":"repeat(8,1fr)",
                         "gap":"8px","padding":"14px 24px 0"}, children=[
            kpi("Mean η 2025",    f"{stats['eta_act_25']:.4f}",  C25,  f"{stats['n25']} pts"),
            kpi("Mean η 2026",    f"{stats['eta_act_26']:.4f}",  C26,  f"{stats['n26']} pts"),
            kpi("Degradation Δη", f"{stats['delta_eta_mean']:.4f}", RED, "2026 − 2025"),
            kpi("Degradation %",  delta_eta_pct_str,              RED,  "all 31 bins worse"),
            kpi("η gap 2025",     f"{stats['eta_err_25']:.2f}%", C25,  "vs hill chart"),
            kpi("η gap 2026",     f"{stats['eta_err_26']:.2f}%", C26,  "vs hill chart"),
            kpi("Flow err 2025",  f" +{stats['q_err_25']:.2f}%",  C25,  "actual > expected"),
            kpi("Flow err 2026",  f" +{stats['q_err_26']:.2f}%",  C26,  f"Δ +{q_delta:.2f}%"),
        ]),

        # ── Plot buttons ────────────────────────────────────────────
        html.Div(style={"padding":"14px 24px 8px"}, children=[
            html.Div("Select plot:", style={"fontSize":"10px","color":MUTED,
                                             "textTransform":"uppercase",
                                             "letterSpacing":"0.06em","marginBottom":"10px"}),
            html.Div(style={"display":"flex","gap":"8px","flexWrap":"wrap"}, children=[
                html.Button(
                    [html.Span(icon, style={"fontSize":"16px","display":"block","marginBottom":"3px"}),
                     html.Span(label, style={"fontSize":"10px"})],
                    id={"type":"pltbtn","index":key}, n_clicks=0,
                    style={"background":CARD2,"border":f"1px solid {GRID}","borderRadius":"8px",
                           "color":MUTED,"cursor":"pointer","fontFamily":"Courier New, monospace",
                           "padding":"9px 13px","textAlign":"center","minWidth":"85px"}
                ) for key, (icon, label) in PLOTS.items()
            ])
        ]),

        # ── Chart area ──────────────────────────────────────────────
        html.Div(style={"margin":"8px 24px 0","background":CARD,"border":f"1px solid {GRID}",
                         "borderRadius":"10px","padding":"14px"}, children=[
            html.Div(id="chart-title",
                     style={"fontSize":"11px","color":MUTED,"marginBottom":"8px","fontStyle":"italic"}),
            dcc.Graph(id="main-graph",
                      config={"displayModeBar":True,"displaylogo":False,
                              "modeBarButtonsToRemove":["lasso2d","select2d"]},
                      style={"height":"420px"}),
            html.Div(id="stats-panel"),
        ]),

        # ── Degradation table ───────────────────────────────────────
        html.Div(style={"margin":"10px 24px 0","background":CARD,"border":f"1px solid {GRID}",
                         "borderRadius":"10px","padding":"14px","overflowX":"auto"}, children=[
            html.Div("🌡️  Δη per Head × Power Bin (2026 − 2025)  |  Red = degraded  |  All 31 common bins show degradation",
                     style={"fontSize":"11px","color":MUTED,"marginBottom":"10px"}),
            html.Table([html.Thead(tbl_header), html.Tbody(tbl_rows)],
                       style={"borderCollapse":"collapse","width":"100%","fontSize":"10px"}),
            html.Div([
                html.Span("■ Δη > 0  ", style={"color":GREEN,"marginRight":"12px"}),
                html.Span("■ −0.010 to 0  ", style={"color":AMBER,"marginRight":"12px"}),
                html.Span("■ < −0.010  ", style={"color":RED}),
            ], style={"marginTop":"8px","fontSize":"10px"}),
        ]),

        html.Div("Unit 2 · LOKA Hill Chart · LinearNDInterpolator · "
                 "2025 (Feb 10 – Mar 17) vs 2026 (Jan 28 – Feb 26)",
                 style={"padding":"12px 24px","fontSize":"9px","color":MUTED}),
    ])

    # ── Callback ─────────────────────────────────────────────────────
    @app.callback(
        dash.Output("main-graph",  "figure"),
        dash.Output("stats-panel", "children"),
        dash.Output("chart-title", "children"),
        [dash.Input({"type":"pltbtn","index":k}, "n_clicks") for k in PLOTS],
        prevent_initial_call=False,
    )
    def render(*args):
        ctx = dash.callback_context
        key = list(PLOTS.keys())[0]
        if ctx.triggered and ctx.triggered[0]["prop_id"] != ".":
            raw = ctx.triggered[0]["prop_id"]
            key = raw.split('"index":"')[1].split('"')[0]
        fig, st = run_plot(key, df25, df26, delta, delta_q, stats)
        icon, label = PLOTS[key]
        return fig, st, f"{icon}  {label}"

    return app

# ═══════════════════════════════════════════════════════════════════
# 8. MAIN
# ═══════════════════════════════════════════════════════════════════
if __name__ == "__main__":
    df25, df26, combined, delta, delta_q, hill, stats = load_all()
    app = build_app(df25, df26, delta, delta_q, stats)
    print("=" * 55)
    print("  Unit 2 Degradation Dashboard")
    print("  Open:  http://127.0.0.1:8050")
    print("=" * 55)
    app.run(debug=False, port=8050)

Parsing hill chart...
Processing 2025...
  1184 pts | η_act=0.9265 | η_err=-2.421%
Processing 2026...
  896 pts | η_act=0.9134 | η_err=-3.532%

Degradation: Δη = -0.0109 (-1.18%)
All 31 bins degraded, 0 improved
  Unit 2 Degradation Dashboard
  Open:  http://127.0.0.1:8050
Dash is running on http://127.0.0.1:8050/



INFO:dash.dash:Dash is running on http://127.0.0.1:8050/



 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:8050
INFO:werkzeug:Press CTRL+C to quit
